In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import json
import re
from tqdm import tqdm
from pathlib import Path
import random

from ReasoningGraph import ReasoningGraph

## Generate Rollouts

In [3]:
# model_name = "Qwen/Qwen3-4B-Thinking-2507"
# model_name = "Qwen/Qwen3-1.7B"
model_name = "Qwen/Qwen2.5-Math-1.5B-Instruct"
num_problems = 2
num_rollouts = 3
temperature = 1.0

In [ ]:
def extract_answer(text):
    """Extract the final numerical answer from the model output."""
    # Look for patterns like "####" (GSM8K format) or boxed answers

    # Pattern 1: #### X format (GSM8K style)
    boxed_match = re.search(r'####\s*(-?\d+(?:\.\d+)?)', text)
    if boxed_match:
        return float(boxed_match.group(1))

    # Pattern 2: \boxed{X} format
    boxed_match = re.search(r'\\boxed\{(-?\d+(?:\.\d+)?)\}', text)
    if boxed_match:
        return float(boxed_match.group(1))

    # Pattern 3: "The answer is X" or "Therefore, X"
    answer_patterns = [
        r'[Tt]he answer is\s*\$?(-?\d+(?:\.\d+)?)',
        r'[Tt]herefore,?\s*\$?(-?\d+(?:\.\d+)?)',
        r'[Ss]o the answer is\s*\$?(-?\d+(?:\.\d+)?)',
    ]

    for pattern in answer_patterns:
        match = re.search(pattern, text)
        if match:
            return float(match.group(1))

    # Pattern 4: Last number in the text
    numbers = re.findall(r'(-?\d+(?:\.\d+)?)', text)
    if numbers:
        return float(numbers[-1])

    return None

def load_problems_from_dataset(dataset='openai/gsm8k', num_problems=10, split='test', seed=None):
    """Load a subset of problems from the provided dataset.
    
    Args:
        dataset: Name of HuggingFace dataset from which to sample problems
        num_problems: Number of problems to sample
        split: Dataset split ('train' or 'test')
        seed: Random seed for reproducibility. If None, sampling will be truly random.
    """
    
    print(f"Loading {num_problems} problems from {dataset}:{split}")
    
    dataset = load_dataset(dataset, "main")
    data = dataset[split]
    
    # Only set seed if one is provided
    if seed is not None:
        random.seed(seed)
        print(f"Using fixed seed: {seed}")
    
    # Sample without replacement
    total_problems = len(data)
    if num_problems > total_problems:
        print(f"Warning: Requested {num_problems} problems but dataset only has {total_problems}.")
        num_problems = total_problems
    
    indices = random.sample(range(total_problems), num_problems)
    
    # Reset seed if we set it
    if seed is not None:
        random.seed()

    problems = []
    for idx in indices:
        item = data[idx]
        # Extract ground truth answer from GSM8K format (after ####)
        gt_match = re.search(r'####\s*(-?\d+(?:\.\d+)?)', item['answer'])
        ground_truth = float(gt_match.group(1)) if gt_match else None

        problems.append({
            'index': idx,
            'question': item['question'],
            'full_answer': item['answer'],
            'ground_truth': ground_truth
        })

    return problems

dataset = 'openai/gsm8k'
problems = load_problems_from_dataset(num_problems=num_problems)

for i, p in enumerate(problems[:3]):
    print(f"\nProblem {i+1}: {p['question'][:100]}...")
    print(f"Ground truth: {p['ground_truth']}")

Loading 2 problems from GSM8k

Problem 1: Fred was preparing for a party to be held in four days.  So, he made 24 gallons of root beer on the ...
Ground truth: 2.0

Problem 2: Artemis is potting flowers with her father. They buy a 30-pound bag of soil. Each rose needs 1 pound...
Ground truth: 3.0

Problem 1: Fred was preparing for a party to be held in four days.  So, he made 24 gallons of root beer on the ...
Ground truth: 2.0

Problem 2: Artemis is potting flowers with her father. They buy a 30-pound bag of soil. Each rose needs 1 pound...
Ground truth: 3.0


In [7]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto"
)

In [ ]:
from experiment import Experiment

# Create an experiment
experiment = Experiment(model_name, problems, num_rollouts, temperature)

# Setup the experiment
experiment.setup()

# Conduct the experiment with initiailized model and tokenizer
experiment.conduct_experiment(model, tokenizer)

Created experiment at: outputs/experiment_20251027_142802


In [ ]:
def conduct_experiment():
    # Run experiments for all problems
    for prob_idx, problem in enumerate(problems):
        print(f"\nProblem {prob_idx + 1}/{len(problems)}")
        
        # Generate rollouts for this problem
        reasoning_graph = ReasoningGraph()
        rollouts = generate_rollouts(
            model, 
            tokenizer, 
            reasoning_graph, 
            problem, 
            num_rollouts=num_rollouts, 
            temperature=temperature
        )
        
        # Save each rollout
        for rollout_idx, rollout_data in enumerate(rollouts):
            experiment.save_rollout(prob_idx, rollout_idx, reasoning_graph, rollout_data)

    # Save final results
    experiment.finalize()

    print("\nExperiment completed! Results saved to:", experiment.base_dir)


Problem 1/2



Problem 2/2



Experiment completed! Results saved to: outputs/experiment_20251027_135904


In [ ]:
# Example: Load and analyze experiment results
def load_experiment_results(experiment_dir):
    """Load results from a saved experiment file (experiment.json)."""
    base_dir = Path(experiment_dir)

    with open(base_dir / "experiment.json") as f:
        data = json.load(f)

    config = data.get('config', {})
    results = data.get('results', [])

    return config, results

# You can load your experiment results like this:
config, results = load_experiment_results(experiment.base_dir)
print("\nExperiment Summary (computed from results):")
print(f"Total rollouts: {len(results)}")
if len(results) > 0:
    accuracy = sum(1 for r in results if r.get('is_correct', False)) / len(results)
    avg_tokens = sum(r.get('num_tokens', 0) for r in results) / len(results)
    avg_entropy = sum(r.get('entropy_mean', 0) for r in results) / len(results)
else:
    accuracy = avg_tokens = avg_entropy = 0

print(f"Accuracy: {accuracy:.2%}")
print(f"Average tokens per generation: {avg_tokens:.1f}")
print(f"Average entropy: {avg_entropy:.3f}")


Experiment Summary:
Total rollouts: 40
Accuracy: 42.50%
Average tokens per generation: 325.2
Average entropy: 0.148


## Saving ReasoningGraph object in a file

In [31]:
# Save the reasoning graph for this generation
from pathlib import Path

# Create outputs directory if it doesn't exist
output_dir = Path("outputs") / f"problem_{prob_idx}_generation"
reasoning_graph.save(output_dir)

print(f"Saved reasoning graph to {output_dir}")

Saved reasoning graph to outputs/problem_4_generation


## Visualizing Entropy across Generated Tokens

In [23]:
from IPython.display import HTML, Markdown
import numpy as np

def visualize_tokens(tokens, entropies):
    """Visualize tokens with color intensity based on entropy."""
    entropies = np.array(entropies)
    # Normalize to 0-1 range
    norm_entropies = (entropies - entropies.min()) / (entropies.max() - entropies.min() + 1e-8)
    
    html = '<div style="line-height: 2; font-family: monospace; font-size: 14px;">'
    for token, intensity in zip(tokens, norm_entropies):
        # Escape HTML characters
        token = token.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
        html += f'<span style="background-color: rgba(100, 150, 255, {intensity:.2f});">{token}</span>'
    html += '</div>'
    
    return HTML(html)

In [24]:
tokens = [tokenizer.decode(id) for id in output_ids[len(model_inputs.input_ids[0]):]]
entropies = reasoning_graph.metrics
visualize_tokens(tokens, entropies)

In [25]:
# Example of loading a saved reasoning graph
loaded_graph = ReasoningGraph.load(output_dir)

# Verify the loaded data
print("Metrics length:", len(loaded_graph.metrics))
print("Probability distributions shape:", loaded_graph.prob_distributions[0].shape)
print("Node cutoff value:", loaded_graph.node_cutoff)

# You can now use this loaded graph for visualization or analysis

Metrics length: 318
Probability distributions shape: torch.Size([151936])
Node cutoff value: 0.058349609375
